# Stage 6: Vector Space Model and Inverted Index

### Project Structure

```bash
CS5246Project/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │    
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                       
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
    ├─ Stage_4_Singlish_to_English_Conversion.ipynb     
    ├─ Stage_5_Common_Normalisation.ipynb
-----------------------------------------------------------------
│   ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb      │
-----------------------------------------------------------------
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb rank_bm25 sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.7 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=39e8a7f1b958a1f4a738de46799454194495f28233d24cbcafccdcc2a630fae2
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


### Change Directory

In [ ]:
# Path
# ----
FOLDER_PATH: str = "/content/drive/MyDrive/CS5246Project/"

import os
os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import os
import re
import json
import joblib           # Use for saving TF-IDF Model
import import_ipynb
import pandas as pd
from rank_bm25 import BM25Okapi
from Stage_2_POS_and_NER_Tagging import pos_tagger
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import lil_matrix, csr_matrix, save_npz, load_npz
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, output_base_dir, vector_database_dir, models_dir, inverted_index_dir

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Vector Space Model

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/CommentVault.csv', keep_default_na=False)

### TF-IDF (Term Frequency–Inverse Document Frequency)

In [ ]:
# ----------------
# Posts Vectorizer
# ----------------
tfidf_posts_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_posts_vectorizer.fit(df_posts['lemmatized_full_text'])

tfidf_titles_matrix = tfidf_posts_vectorizer.transform(df_posts['lemmatized_title'])
tfidf_contents_matrix = tfidf_posts_vectorizer.transform(df_posts['lemmatized_content'])
tfidf_full_text_matrix = tfidf_posts_vectorizer.transform(df_posts['lemmatized_full_text'])

print("TF-IDF vectorization completed for posts")
print("----------------------------------------")
print(f"Shape of TF-IDF matrix for title: {tfidf_titles_matrix.shape}")
print(f"Shape of TF-IDF matrix for content: {tfidf_contents_matrix.shape}")
print(f"Shape of TF-IDF matrix for full text: {tfidf_full_text_matrix.shape}")
print()

# -------------------
# Comments Vectorizer
# -------------------
tfidf_comments_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_comments_matrix = tfidf_comments_vectorizer.fit_transform(df_comments['cleaned_text'])

print("TF-IDF vectorization completed for posts")
print("----------------------------------------")
print(f"Shape of TF-IDF matrix for comments: {tfidf_comments_matrix.shape}")

TF-IDF vectorization completed for posts
----------------------------------------
Shape of TF-IDF matrix for title: (31957, 28931)
Shape of TF-IDF matrix for content: (31957, 28931)
Shape of TF-IDF matrix for full text: (31957, 28931)

TF-IDF vectorization completed for posts
----------------------------------------
Shape of TF-IDF matrix for comments: (764462, 148690)


### Save TF-IDF Matrices

In [ ]:
# ---------
# Data Path
# ---------
tfidf_titles_path = os.path.join(vector_database_dir, 'tfidf_titles.npz')
tfidf_contents_path = os.path.join(vector_database_dir, 'tfidf_contents.npz')
tfidf_full_text_path = os.path.join(vector_database_dir, 'tfidf_fulltext.npz')

tfidf_comments_path = os.path.join(vector_database_dir, 'tfidf_comments.npz')

# --------------------
# Save TF-IDF Matrices
# --------------------
save_npz(tfidf_titles_path, tfidf_titles_matrix)
save_npz(tfidf_contents_path, tfidf_contents_matrix)
save_npz(tfidf_full_text_path, tfidf_full_text_matrix)

save_npz(tfidf_comments_path, tfidf_comments_matrix)

print(f"TF-IDF titles matrix saved to: {tfidf_titles_path}")
print(f"TF-IDF contents matrix saved to: {tfidf_contents_path}")
print(f"TF-IDF full text matrix saved to: {tfidf_full_text_path}")

print(f"TF-IDF comments matrix saved to: {tfidf_comments_path}")

TF-IDF titles matrix saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_titles.npz
TF-IDF contents matrix saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_contents.npz
TF-IDF full text matrix saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_fulltext.npz
TF-IDF comments matrix saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_comments.npz


### Save TF-IDF Vectorizer

In [ ]:
# ---------
# Data Path
# ---------
tfidf_posts_vectorizer_path = os.path.join(models_dir, 'tfidf_posts_vectorizer.joblib')
tfidf_comments_vectorizer_path = os.path.join(models_dir, 'tfidf_comments_vectorizer.joblib')

# ----------------------
# Save TF-IDF Vectorizer
# ----------------------
joblib.dump(tfidf_posts_vectorizer, tfidf_posts_vectorizer_path)
joblib.dump(tfidf_comments_vectorizer, tfidf_comments_vectorizer_path)

print(f"TF-IDF posts vectorizer saved to: {tfidf_posts_vectorizer_path}")
print(f"TF-IDF comments vectorizer saved to: {tfidf_comments_vectorizer_path}")

TF-IDF posts vectorizer saved to: /content/drive/MyDrive/CS5246Project/data/models/tfidf_posts_vectorizer.joblib
TF-IDF comments vectorizer saved to: /content/drive/MyDrive/CS5246Project/data/models/tfidf_comments_vectorizer.joblib


### Existence Confirmation

In [ ]:
# Confirm Existence of TF-IDF Titles Matrix
# -----------------------------------------
if os.path.exists(tfidf_titles_path):
    print(f"Confirmation: TF-IDF titles matrix successfully saved at {tfidf_titles_path}")

else:
    print(f"Error: TF-IDF titels matrix NOT found at {tfidf_titles_path}")

# Confirm Existence of TF-IDF Contents Matrix
# -------------------------------------------
if os.path.exists(tfidf_contents_path):
    print(f"Confirmation: TF-IDF contents matrix successfully saved at {tfidf_contents_path}")

else:
    print(f"Error: TF-IDF contents matrix NOT found at {tfidf_contents_path}")

# Confirm Existence of TF-IDF Full Text Matrix
# --------------------------------------------
if os.path.exists(tfidf_full_text_path):
    print(f"Confirmation: TF-IDF full text matrix successfully saved at {tfidf_full_text_path}")

else:
    print(f"Error: TF-IDF full text matrix NOT found at {tfidf_full_text_path}")

# Confirm Existence of TF-IDF Comments Matrix
# -------------------------------------------
if os.path.exists(tfidf_comments_path):
    print(f"Confirmation: TF-IDF comments matrix successfully saved at {tfidf_comments_path}")

else:
    print(f"Error: TF-IDF comments matrix NOT found at {tfidf_comments_path}")

# Confirm Existence of TF-IDF Posts Vectorizer
# --------------------------------------------
if os.path.exists(tfidf_posts_vectorizer_path):
    print(f"Confirmation: TF-IDF post vectorizer successfully saved at {tfidf_posts_vectorizer_path}")

else:
    print(f"Error: TF-IDF post vectorizer NOT found at {tfidf_posts_vectorizer_path}")

# Confirm Existence of TF-IDF Comments Vectorizer
# -----------------------------------------------
if os.path.exists(tfidf_comments_vectorizer_path):
    print(f"Confirmation: TF-IDF comment vectorizer successfully saved at {tfidf_comments_vectorizer_path}")

else:
    print(f"Error: TF-IDF comment vectorizer NOT found at {tfidf_comments_vectorizer_path}")

Confirmation: TF-IDF titles matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_titles.npz
Confirmation: TF-IDF contents matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_contents.npz
Confirmation: TF-IDF full text matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_fulltext.npz
Confirmation: TF-IDF comments matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_comments.npz
Confirmation: TF-IDF post vectorizer successfully saved at /content/drive/MyDrive/CS5246Project/data/models/tfidf_posts_vectorizer.joblib
Confirmation: TF-IDF comment vectorizer successfully saved at /content/drive/MyDrive/CS5246Project/data/models/tfidf_comments_vectorizer.joblib


## BM25 (Best Matching 25)

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/CommentVault.csv', keep_default_na=False)

#### Tokenization

In [ ]:
tokenized_titles = [doc.split(" ") for doc in df_posts['lemmatized_title'].astype(str)]
tokenized_contents = [doc.split(" ") for doc in df_posts['lemmatized_content'].astype(str)]
tokenized_full_text = [doc.split(" ") for doc in df_posts['lemmatized_full_text'].astype(str)]

tokenized_comments = [doc.split(" ") for doc in df_comments['cleaned_text'].astype(str)]

print("Tokenization completed for posts and comments")

Tokenization completed for posts and comments


#### BM25 Model

In [ ]:
# BM25 Posts
bm25_titles = BM25Okapi(tokenized_titles)
bm25_fulltext = BM25Okapi(tokenized_full_text)

# BM25 Comments
bm25_comments = BM25Okapi(tokenized_comments)

print("BM25 models initialized for posts and comments")

BM25 models initialized for posts and comments


### Save BM25 Model

In [ ]:
# Model Path
# ----------
bm25_titles_path = os.path.join(models_dir, 'bm25_titles_model.joblib')
bm25_fulltext_path = os.path.join(models_dir, 'bm25_fulltext_model.joblib')
bm25_comments_path = os.path.join(models_dir, 'bm25_comments_model.joblib')

# Save Model
# ----------
joblib.dump(bm25_titles, bm25_titles_path)
joblib.dump(bm25_fulltext, bm25_fulltext_path)
joblib.dump(bm25_comments, bm25_comments_path)

print(f"BM25 titles model saved to: {bm25_titles_path}")
print(f"BM25 fulltext model saved to: {bm25_fulltext_path}")
print(f"BM25 comments model saved to: {bm25_comments_path}")

BM25 titles model saved to: /content/drive/MyDrive/CS5246Project/data/models/bm25_titles_model.joblib
BM25 fulltext model saved to: /content/drive/MyDrive/CS5246Project/data/models/bm25_fulltext_model.joblib
BM25 comments model saved to: /content/drive/MyDrive/CS5246Project/data/models/bm25_comments_model.joblib


### Generate BM25 Matrices

In [ ]:
# Obtain the complete vocabulary from the bm25_posts model's idf attribute keys.
# Create a mapping from each term in this vocabulary to a unique integer index.
title_vocab = set()
title_vocab.update(*tokenized_titles)
title_term_to_idx = {term: idx for idx, term in enumerate(title_vocab)}

fulltext_vocab = set()
fulltext_vocab.update(*tokenized_full_text)
fulltext_term_to_idx = {term: idx for idx, term in enumerate(fulltext_vocab)}

comment_vocab = set()
comment_vocab.update(*tokenized_comments)
comment_term_to_idx = {term: idx for idx, term in enumerate(comment_vocab)}

print("---------")
print("PostVault")
print("---------")
print("Title:")
print(f"Vocabulary size: {len(title_vocab)}")
print("Term-to-index mapping created.")
print()
print("Full Text:")
print(f"Vocabulary size: {len(fulltext_vocab)}")
print("Term-to-index mapping created.")
print()
print("------------")
print("CommentVault")
print("------------")
print(f"Vocabulary size: {len(comment_vocab)}")
print("Term-to-index mapping created.")

---------
PostVault
---------
Title:
Vocabulary size: 19869
Term-to-index mapping created.

Full Text:
Vocabulary size: 29188
Term-to-index mapping created.

------------
CommentVault
------------
Vocabulary size: 149032
Term-to-index mapping created.


In [ ]:
from collections import Counter
from scipy.sparse import lil_matrix

def build_bm25_matrix_from_external_docs(tokenized_docs, bm25_model, term_to_idx):
    num_docs = len(tokenized_docs)
    vocab_size = len(term_to_idx)

    matrix = lil_matrix((num_docs, vocab_size), dtype=float)

    avg_dl = sum(bm25_model.doc_len) / len(bm25_model.doc_len)

    for i, doc_tokens in enumerate(tokenized_docs):
        tf = Counter(doc_tokens)

        doc_length = len(doc_tokens)

        for term, freq in tf.items():
            if term in term_to_idx:
                term_index = term_to_idx[term]
                idf = bm25_model.idf.get(term, 0)

                numerator = freq * (bm25_model.k1 + 1)
                denominator = freq + bm25_model.k1 * (
                    1 - bm25_model.b + bm25_model.b * (doc_length / avg_dl)
                )

                score = idf * (numerator / denominator) if denominator != 0 else 0

                matrix[i, term_index] = score

    return matrix.tocsr()

In [ ]:
bm25_titles_matrix = build_bm25_matrix_from_external_docs(
    tokenized_titles, bm25_titles, title_term_to_idx
)
print("-----------------")
print("BM25 Title Matrix")
print("-----------------")
print("Shape:", bm25_titles_matrix.shape)
print()

bm25_contents_matrix = build_bm25_matrix_from_external_docs(
    tokenized_contents, bm25_fulltext, fulltext_term_to_idx
)

print("--------------------")
print("BM25 Contents Matrix")
print("--------------------")
print("Shape:", bm25_contents_matrix.shape)
print()

bm25_full_text_matrix = build_bm25_matrix_from_external_docs(
    tokenized_full_text, bm25_fulltext, fulltext_term_to_idx
)

print("---------------------")
print("BM25 Full Text Matrix")
print("---------------------")
print("Shape:", bm25_full_text_matrix.shape)
print()

bm25_comments_matrix = build_bm25_matrix_from_external_docs(
    tokenized_comments, bm25_comments, comment_term_to_idx
)
print("--------------------")
print("BM25 Comments Matrix")
print("--------------------")
print("Shape:", bm25_comments_matrix.shape)
print()


-----------------
BM25 Title Matrix
-----------------
Shape: (31957, 19869)

--------------------
BM25 Contents Matrix
--------------------
Shape: (31957, 29188)

---------------------
BM25 Full Text Matrix
---------------------
Shape: (31957, 29188)

--------------------
BM25 Comments Matrix
--------------------
Shape: (764462, 149032)



### Save BM25 Matrices

In [ ]:
from scipy.sparse import save_npz


bm25_titles_matrix_path = os.path.join(vector_database_dir, 'bm25_titles.npz')
bm25_contents_matrix_path = os.path.join(vector_database_dir, 'bm25_contents.npz')
bm25_full_text_matrix_path = os.path.join(vector_database_dir, 'bm25_fulltext.npz')

bm25_comments_matrix_path = os.path.join(vector_database_dir, 'bm25_comments.npz')

# ------------------
# Save BM25 Matrices
# ------------------
save_npz(bm25_titles_matrix_path, bm25_titles_matrix)
save_npz(bm25_contents_matrix_path, bm25_contents_matrix)
save_npz(bm25_full_text_matrix_path, bm25_full_text_matrix)
save_npz(bm25_comments_matrix_path, bm25_comments_matrix)

### Existence Confirmation

In [ ]:
# Model Existence
# ---------------
if os.path.exists(bm25_titles_path):
    print(f"Confirmation: BM25 titles model successfully saved at {bm25_titles_path}")

else:
    print(f"Error: BM25 titles model NOT found at {bm25_titles_path}")

if os.path.exists(bm25_fulltext_path):
    print(f"Confirmation: BM25 full text model successfully saved at {bm25_fulltext_path}")

else:
    print(f"Error: BM25 full text model NOT found at {bm25_fulltext_path}")

if os.path.exists(bm25_comments_path):
    print(f"Confirmation: BM25 comments model successfully saved at {bm25_comments_path}")

else:
    print(f"Error: BM25 comments model NOT found at {bm25_comments_path}")

# Data Existence
# --------------
if os.path.exists(bm25_titles_matrix_path):
    print(f"Confirmation: BM25 titles matrix successfully saved at {bm25_titles_matrix_path}")

else:
    print(f"Error: BM25 titles matrix NOT found at {bm25_titles_matrix_path}")

if os.path.exists(bm25_contents_matrix_path):
    print(f"Confirmation: BM25 contents matrix successfully saved at {bm25_contents_matrix_path}")

else:
    print(f"Error: BM25 contents matrix NOT found at {bm25_contents_matrix_path}")

if os.path.exists(bm25_full_text_matrix_path):
    print(f"Confirmation: BM25 full text matrix successfully saved at {bm25_full_text_matrix_path}")

else:
    print(f"Error: BM25 full text matrix NOT found at {bm25_full_text_matrix_path}")

if os.path.exists(bm25_comments_matrix_path):
    print(f"Confirmation: BM25 comments matrix successfully saved at {bm25_comments_matrix_path}")

else:
    print(f"Error: BM25 comments matrix NOT found at {bm25_comments_matrix_path}")

Confirmation: BM25 titles model successfully saved at /content/drive/MyDrive/CS5246Project/data/models/bm25_titles_model.joblib
Confirmation: BM25 full text model successfully saved at /content/drive/MyDrive/CS5246Project/data/models/bm25_fulltext_model.joblib
Confirmation: BM25 comments model successfully saved at /content/drive/MyDrive/CS5246Project/data/models/bm25_comments_model.joblib
Confirmation: BM25 titles matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bm25_titles.npz
Confirmation: BM25 contents matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bm25_contents.npz
Confirmation: BM25 full text matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bm25_fulltext.npz
Confirmation: BM25 comments matrix successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bm25_comments.npz


## Document Bert

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/CommentVault.csv', keep_default_na=False)

/tmp/ipykernel_55063/4176693567.py:4: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df_posts = pd.read_csv('/content/drive/MyDrive/CS5246Project/data/PostVault.csv', keep_default_na=False)


### Load Pre-trained BERT Model

In [ ]:
# Load a Pre-trained Sentence Transformer Model
# ---------------------------------------------
# 'all-MiniLM-L6-v2' is a good balance of performance and speed
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Pre-trained Sentence-BERT model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pre-trained Sentence-BERT model loaded successfully.


### Generate Embeddings

In [ ]:
# -----------------------------------
# Generate embeddings for Post Titles
# -----------------------------------
print("----------------------------------------")
print("Generating Embeddings for Post Titles...")
print("----------------------------------------")
bert_titles_embeddings = bert_model.encode(df_posts['title'].tolist(), show_progress_bar=True)
print(f"Shape of BERT embeddings for titles: {bert_titles_embeddings.shape}\n")
print()

# ------------------------------------
# Generate Embeddings for Post Content
# ------------------------------------
print("-----------------------------------------")
print("Generating Embeddings for Post Content...")
print("-----------------------------------------")
bert_contents_embeddings = bert_model.encode(df_posts['content'].tolist(), show_progress_bar=True)
print(f"Shape of BERT embeddings for content: {bert_contents_embeddings.shape}\n")
print()

# --------------------------------------
# Generate Embeddings for Post Full Text
# --------------------------------------
print("-------------------------------------------")
print("Generating Embeddings for Post Full Text...")
print("-------------------------------------------")
bert_full_text_embeddings = bert_model.encode(df_posts['fulltext'].tolist(), show_progress_bar=True)
print(f"Shape of BERT embeddings for full text: {bert_full_text_embeddings.shape}\n")
print()

# --------------------------------
# Generate Embeddings for Comments
# --------------------------------
# print("-------------------------------------")
# print("Generating Embeddings for Comments...")
# print("-------------------------------------")
# bert_comments_embeddings = bert_model.encode(df_comments['cleaned_text'].tolist(), show_progress_bar=True)
# print(f"Shape of BERT embeddings for comments: {bert_comments_embeddings.shape}\n")
# print()

# print("BERT embeddings generation completed.")

----------------------------------------
Generating Embeddings for Post Titles...
----------------------------------------


Batches:   0%|          | 0/999 [00:00<?, ?it/s]

Shape of BERT embeddings for titles: (31957, 384)


-----------------------------------------
Generating Embeddings for Post Content...
-----------------------------------------


Batches:   0%|          | 0/999 [00:00<?, ?it/s]

Shape of BERT embeddings for content: (31957, 384)


-------------------------------------------
Generating Embeddings for Post Full Text...
-------------------------------------------


Batches:   0%|          | 0/999 [00:00<?, ?it/s]

Shape of BERT embeddings for full text: (31957, 384)




### Save Embeddings

In [ ]:
import numpy as np

# Bert Embedding Path
# -------------------
## Posts
## -----
bert_titles_embeddings_path = os.path.join(vector_database_dir, 'bert_titles_embeddings.npy')
bert_contents_embeddings_path = os.path.join(vector_database_dir, 'bert_contents_embeddings.npy')
bert_full_text_embeddings_path = os.path.join(vector_database_dir, 'bert_fulltext_embeddings.npy')


## Comments
## --------
# bert_comments_embeddings_path = os.path.join(vector_database_dir, 'bert_comments_embeddings.npy')

# Save Embeddings
# ---------------
## Posts
## -----
np.save(bert_titles_embeddings_path, bert_titles_embeddings)
np.save(bert_contents_embeddings_path, bert_contents_embeddings)
np.save(bert_full_text_embeddings_path, bert_full_text_embeddings)
print(f"BERT Titles Embeddings saved to: {bert_titles_embeddings_path}")
print(f"BERT Contents Embeddings saved to: {bert_contents_embeddings_path}")
print(f"BERT Full Text Embeddings saved to: {bert_full_text_embeddings_path}")

## Comments
## --------
# np.save(bert_comments_embeddings_path, bert_comments_embeddings)
# print(f"BERT Comments Embeddings saved to: {bert_comments_embeddings_path}")

BERT Titles Embeddings saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/bert_titles_embeddings.npy
BERT Contents Embeddings saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/bert_contents_embeddings.npy
BERT Full Text Embeddings saved to: /content/drive/MyDrive/CS5246Project/data/vector_database/bert_fulltext_embeddings.npy


### Save BERT Model

In [ ]:
bert_model_path = os.path.join(models_dir, 'sentence_bert_model')
bert_model.save(bert_model_path)

print("-" * 70)
print(f"Sentence-BERT model saved to: {bert_model_path}")
print("-" * 70)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

----------------------------------------------------------------------
Sentence-BERT model saved to: /content/drive/MyDrive/CS5246Project/data/models/sentence_bert_model
----------------------------------------------------------------------


### Existence Confirmation

In [ ]:
# -----------------------------------------------
# Confirm Existence of BERT Embeddings for Titles
# -----------------------------------------------
if os.path.exists(bert_titles_embeddings_path):
    print(f"Confirmation: BERT titles embeddings successfully saved at {bert_titles_embeddings_path}")

else:
    print(f"Error: BERT titles embeddings NOT found at {bert_titles_embeddings_path}")

# -------------------------------------------------
# Confirm Existence of BERT Embeddings for Contents
# -------------------------------------------------
if os.path.exists(bert_contents_embeddings_path):
    print(f"Confirmation: BERT contents embeddings successfully saved at {bert_contents_embeddings_path}")

else:
    print(f"Error: BERT contents embeddings NOT found at {bert_contents_embeddings_path}")

# --------------------------------------------------
# Confirm Existence of BERT Embeddings for Full Text
# --------------------------------------------------
if os.path.exists(bert_full_text_embeddings_path):
    print(f"Confirmation: BERT full text embeddings successfully saved at {bert_full_text_embeddings_path}")

else:
    print(f"Error: BERT full text embeddings NOT found at {bert_full_text_embeddings_path}")

# -------------------------------------------------
# Confirm Existence of BERT Embeddings for Comments
# -------------------------------------------------
# if os.path.exists(bert_comments_embeddings_path):
#     print(f"Confirmation: BERT comments embeddings successfully saved at {bert_comments_embeddings_path}")

# else:
#     print(f"Error: BERT comments embeddings NOT found at {bert_comments_embeddings_path}")

# -------------------------------
# Confirm Existence of BERT Model
# -------------------------------
if os.path.exists(bert_model_path):
    print(f"Confirmation: Sentence-BERT model successfully saved at {bert_model_path}")

else:
    print(f"Error: Sentence-BERT model NOT found at {bert_model_path}")

Confirmation: BERT titles embeddings successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bert_titles_embeddings.npy
Confirmation: BERT contents embeddings successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bert_contents_embeddings.npy
Confirmation: BERT full text embeddings successfully saved at /content/drive/MyDrive/CS5246Project/data/vector_database/bert_fulltext_embeddings.npy
Confirmation: Sentence-BERT model successfully saved at /content/drive/MyDrive/CS5246Project/data/models/sentence_bert_model


## Inverted Index

### Load TF-IDF Vectorizer and Matrices

In [ ]:
# Load TF-IDF Vectorizer
tfidf_posts_vectorizer = joblib.load(tfidf_posts_vectorizer_path)

# Load TF-IDF Matrices
tfidf_titles_matrix = load_npz(tfidf_titles_path)
tfidf_full_text_matrix = load_npz(tfidf_full_text_path)

print("---------")
print("Load Data")
print("---------")
print(f"TF-IDF Titles Matrix loaded from: {tfidf_titles_path}")
print(f"TF-IDF Full Text Matrix loaded from: {tfidf_full_text_path}")
print()
print("---------------")
print("Load Vectorizer")
print("---------------")
print(f"TF-IDF Posts Vectorizer loaded from: {tfidf_posts_vectorizer_path}")

---------
Load Data
---------
TF-IDF Titles Matrix loaded from: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_titles.npz
TF-IDF Full Text Matrix loaded from: /content/drive/MyDrive/CS5246Project/data/vector_database/tfidf_fulltext.npz

---------------
Load Vectorizer
---------------
TF-IDF Posts Vectorizer loaded from: /content/drive/MyDrive/CS5246Project/data/models/tfidf_posts_vectorizer.joblib


### Build Inverted Index Function

In [ ]:
def build_inverted_index(tfidf_matrix, feature_names):
    inverted_index = {}
    for doc_idx, row in enumerate(tfidf_matrix):
        # row is a sparse matrix row, convert to dictionary for easy iteration
        for col_idx, score in zip(row.indices, row.data):
            word = feature_names[col_idx]
            if word not in inverted_index:
                inverted_index[word] = []
            inverted_index[word].append((doc_idx, score))

    # Sort the lists by score in descending order
    for word, postings_list in inverted_index.items():
        inverted_index[word] = sorted(postings_list, key=lambda x: x[1], reverse=True)

    return inverted_index

print("Inverted index building function defined.")

Inverted index building function defined.


### Generate Inverted Indices

In [ ]:
feature_names = tfidf_posts_vectorizer.get_feature_names_out()

print("-------------------------")
print("Inverted Index for Titles")
print("-------------------------")
inverted_index_titles = build_inverted_index(tfidf_titles_matrix, feature_names)
print(f"Inverted index for titles created. Number of terms: {len(inverted_index_titles)}")

print("----------------------------")
print("Inverted Index for Full Text")
print("----------------------------")
inverted_index_full_text = build_inverted_index(tfidf_full_text_matrix, feature_names)
print(f"Inverted index for full text created. Number of terms: {len(inverted_index_full_text)}")

-------------------------
Inverted Index for Titles
-------------------------
Inverted index for titles created. Number of terms: 19641
----------------------------
Inverted Index for Full Text
----------------------------
Inverted index for full text created. Number of terms: 28931


### Save Inverted Indices

In [ ]:
inverted_index_dir = '/content/drive/MyDrive/CS5246Project/data/inverted_index'

# Define paths for saving
inverted_index_titles_path = os.path.join(inverted_index_dir, 'inverted_index_tfidf_titles.json')
inverted_index_full_text_path = os.path.join(inverted_index_dir, 'inverted_index_tfidf_fulltext.json')

# Save the inverted indices as JSON files
with open(inverted_index_titles_path, 'w', encoding='utf-8') as f:
    json.dump(inverted_index_titles, f, ensure_ascii=False, indent=4)

with open(inverted_index_full_text_path, 'w', encoding='utf-8') as f:
    json.dump(inverted_index_full_text, f, ensure_ascii=False, indent=4)

print(f"Inverted index for titles saved to: {inverted_index_titles_path}")
print(f"Inverted index for full text saved to: {inverted_index_full_text_path}")

Inverted index for titles saved to: /content/drive/MyDrive/CS5246Project/data/inverted_index/inverted_index_tfidf_titles.json
Inverted index for full text saved to: /content/drive/MyDrive/CS5246Project/data/inverted_index/inverted_index_tfidf_fulltext.json
